<a href="https://colab.research.google.com/github/nbisoyi3024/lora-finetuning-qwen-alpaca/blob/main/lora_finetuning_alpaca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
# install the libraries
!pip install -q transformers peft datasets accelerate

### Load dataset

In [30]:
from datasets import load_dataset

# load the Alpaca dataset
dataset = load_dataset("tatsu-lab/alpaca", split="train")

# we will keep only 1,100 rows so training will be fast
dataset = dataset.select(range(1100))

# split: 1,000 rows for training and 100 rows for testing
split = dataset.train_test_split(test_size=100, seed=42)
train_data = split["train"]
test_data = split["test"]

# lets check the sizes
print("Training:",len(train_data))
print("Testing:",len(test_data))
print(train_data[0])

Training: 1000
Testing: 100
{'instruction': 'Reverse a linked list.', 'input': '', 'output': 'def reverse(head): \n    prev = None\n    cur = head \n\n    while cur:\n        next = cur.next\n        cur.next = prev \n        prev = cur \n        cur = next\n    head = prev\n    \n    return head', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nReverse a linked list.\n\n### Response:\ndef reverse(head): \n    prev = None\n    cur = head \n\n    while cur:\n        next = cur.next\n        cur.next = prev \n        prev = cur \n        cur = next\n    head = prev\n    \n    return head'}


#### Load the small model

In [31]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

#tokenizer converts texts into numbers
tokenizer = AutoTokenizer.from_pretrained(model_name)

# padding token to make texts same length
tokenizer.pad_token = tokenizer.eos_token

# loads the model and moves  model to colab gpu
model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

print("Model loaded:", model_name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-0.5B


#### step 4: Turn each row into text, then into numbers

In [32]:
# put one row into a single piece of text
def make_text(row):

    # if the row has extra input, include it
    text = f"Instruction: {row['instruction']}\n"

    if row["input"]:
        text += f"Input: {row['input']}\n"

    # add the correct answer and an end marker
    text += f"Answer: {row['output']}{tokenizer.eos_token}"
    return text

#turn the text into numbers
def tokenize(row):
    return tokenizer(
        make_text(row),
        truncation=True,
        max_length=256
    )

# apply it to every row, and drop the old text columns
train_tok = train_data.map(tokenize, remove_columns=train_data.column_names)
test_tok = test_data.map(tokenize, remove_columns=test_data.column_names)

print(len(train_tok), len(test_tok))

1000 100


step 5: Measure the test loss BEFORE training(How well does the original Qwen model perform on my 100 test examples before LoRA fine-tuning?)

In [33]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

args = TrainingArguments(
    output_dir="out",
    per_device_eval_batch_size=8,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    eval_dataset=test_tok,
    data_collator=collator
)

result = trainer.evaluate()

print("Test loss BEFORE training:", result["eval_loss"])

Training Loss,Validation Loss,Step
No log,2.020047,0


Test loss BEFORE training: 2.020047426223755


Step 6: Save 5 sample answers from the untouched model

In [34]:
def ask(instruction):
    prompt = f"Instruction: {instruction}\nAnswer: "

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# take the first 5 questions from the TEST set
test_questions = [test_data[i]["instruction"] for i in range(5)]

# ask the model and save its answers
before_answers = []

for question in test_questions:
    answer = ask(question)
    before_answers.append(answer)

    print("Q:", question)
    print("A:", answer)
    print("-----")

Q: Explain the concept of binary search.
A: 1. Binary search is a search algorithm that works by repeatedly dividing the search space in half until the target value is found or the search space is empty.
2. The algorithm works by starting at the beginning of the search space and comparing the target value to the middle element of the search space. If the target value is less than the middle element, the algorithm searches the left half of the search space. If the target value is greater than the middle element, the algorithm searches the right half of the search space. This process is repeated until the target value is found or the search space is empty.
3. Binary search is an efficient search algorithm because it only requires a constant amount of time to search each half of the search space, regardless of the
-----
Q: Devise a strategic plan for a fashion brand to increase its online presence.
A: 1. Define the target audience: The first step in devising a strategic plan for a fashion

In [35]:
import peft
import transformers

print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)

PEFT: 0.21.1
Transformers: 5.18.0


Step 7: Add LoRA to the model (does not train yet)

r=8 → size of the LoRA adapter
lora_alpha=16 → controls LoRA scaling
lora_dropout=0.05 → helps reduce overfitting
target_modules=["q_proj", "v_proj"] → applies LoRA to the query/value layers
task_type="CAUSAL_LM" → tells PEFT this is a language-generation model

In [36]:

!pip install -q -U peft transformers accelerate

In [37]:
!pip install -U torchao

In [38]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


The model has about 494 million weights. LoRA trains only 540,672 of them, which is 0.11%.

step 8: Train, then measure the AFTER loss

In [39]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="lora_out",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=collator
)

before = result

trainer.train()

after = trainer.evaluate()

print("Test loss BEFORE:", before["eval_loss"])
print("Test loss AFTER:", after["eval_loss"])

Epoch,Training Loss,Validation Loss
1,1.655885,1.715810
2,1.622310,1.707327


Training Loss,Validation Loss,Epoch
1.622310,1.707327,2


Test loss BEFORE: 2.020047426223755
Test loss AFTER: 1.7073273658752441


In [40]:
# ask the same 5 test questions again, now that the model is trained
after_answers = []
for i in range(5):
    q = test_questions[i]
    answer = ask(q)
    after_answers.append(answer)
    print("Q:", q)
    print("BEFORE:", before_answers[i])
    print("AFTER :", answer)
    print("-----")

Q: Explain the concept of binary search.
BEFORE: 1. Binary search is a search algorithm that works by repeatedly dividing the search space in half until the target value is found or the search space is empty.
2. The algorithm works by starting at the beginning of the search space and comparing the target value to the middle element of the search space. If the target value is less than the middle element, the algorithm searches the left half of the search space. If the target value is greater than the middle element, the algorithm searches the right half of the search space. This process is repeated until the target value is found or the search space is empty.
3. Binary search is an efficient search algorithm because it only requires a constant amount of time to search each half of the search space, regardless of the
AFTER :  Binary search is a search algorithm that works by repeatedly dividing the search space in half until the target value is found or the search space is empty. It is 